In [1]:
!pip install autogen-agentchat==0.2.38

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.0/382.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 17.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, b

In [1]:
!pip install ag2[openai,interop-langchain]

  Using cached ag2-0.10.2-py3-none-any.whl.metadata (36 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 922.2/922.2 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [14]:
import autogen
from autogen import ConversableAgent, UserProxyAgent, AssistantAgent, GroupChat, GroupChatManager
from autogen import register_function
from langchain_community.tools import TavilySearchResults
from autogen.interop import Interoperability
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import re
from typing import Any, Dict, Optional
import json


In [2]:
import os
from getpass import getpass

In [3]:
OPENAI_KEY = getpass('Enter Open AI API Key: ')
os.environ["OPENAI_API_KEY"] = OPENAI_KEY

Enter Open AI API Key: ··········


In [40]:
llm_config = {
    "config_list": [{"model": "gpt-4o-mini",
                     "api_key": os.environ["OPENAI_API_KEY"]}],
}

In [41]:
def termination_for(agent_name: str):
    """Limit termination detection to messages from the specific agent."""
    return lambda msg: (
        isinstance(msg, dict)
        and msg.get("name") == agent_name
        and msg.get("content") is not None
        and "TERMINATE" in msg.get("content")
    )


In [42]:
# User proxy: initiates the conversation and terminates on "TERMINATE" or when advisor completes
def user_proxy_termination(msg):
    """Terminate when user says TERMINATE or when investment_advisor_agent completes."""
    if not isinstance(msg, dict):
        return False
    content = msg.get("content", "")
    sender = msg.get("name", "")

    # Stop if user explicitly says TERMINATE
    if "user_proxy" in sender.lower() and "TERMINATE" in content:
        return True

    # Stop if investment_advisor_agent has completed (sent TERMINATE)
    if "investment_advisor_agent" in sender.lower() and "TERMINATE" in content:
        return True

    # Also check if the last message in group_chat is from investment_advisor_agent
    # This handles the case where router returns None but message hasn't been processed yet
    if group_chat.messages:
        last_msg = group_chat.messages[-1]
        if isinstance(last_msg, dict):
            last_sender = last_msg.get("name", "")
            last_content = last_msg.get("content", "")
            if "investment_advisor_agent" in last_sender.lower() and "TERMINATE" in last_content:
                return True

    return False

In [43]:
user_proxy = UserProxyAgent(
    name="user_proxy",
    llm_config=False,
    human_input_mode="NEVER",  # Set to NEVER to prevent auto-reply when no speaker selected
    code_execution_config=False,
    is_termination_msg=user_proxy_termination,
)


# --- Portfolio Agent ---

In [44]:
portfolio_analysis_agent = ConversableAgent(
    name="portfolio_analysis_agent",
    system_message="""
You are the Portfolio Analysis Agent.

Responsibilities:
- Collect and analyze the user's salary and current investments (FDs, SIPs, stocks, real estate, gold, PPF/NPS, cash, debts).
- Infer a risk profile (Conservative | Moderate | Aggressive).
- Decide whether the user should pursue Growth, Value, or Hybrid investing.

Output requirements:
- Salary analysis
- Portfolio strength breakdown
- Risk profile summary
- Recommended investment type (exactly one of: Growth, Value, Hybrid)
- Brief justification

At the end of the message include a line: RECOMMENDATION: <Growth|Value|Hybrid>
Finish the message with TERMINATE.
""",
    description="Analyzes the user's inputs and recommends Growth or Value path.",
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=termination_for("portfolio_analysis_agent"),
)


# --- Growth Agent ---

In [45]:
growth_agent = ConversableAgent(
    name="growth_agent",
    system_message="""
You are the Growth Investment Agent.

Use conversation history to read salary, portfolio summary, risk profile, and recommendation.
If details are missing, ask concise clarifying questions.

Respond in this JSON-like layout (text allowed inside values):
{
  "title": "Growth Plan",
  "recommended_allocation": {"equity_percent": <int>, "debt_percent": <int>, "cash_percent": <int>},
  "top_picks": [
    {"name": "...", "type": "Large-cap MF | Mid-cap MF | Direct Equity | Sectoral Fund", "rationale": "..."},
    {"name": "..."},
    {"name": "..."}
  ],
  "actionable_next_steps": ["..."],
  "risk_notes": "...",
  "explain_short": "1-2 sentence justification"
}

Use Indian market context and keep allocations summing to ~100%.
End with TERMINATE.
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=termination_for("growth_agent"),
)

# --- Value Agent ---

In [46]:
value_agent = ConversableAgent(
    name="value_agent",
    system_message="""
You are the Value Investment Agent.

Use conversation history to read salary, portfolio summary, risk profile, and recommendation.
If details are missing, ask concise clarifying questions.

Respond in this JSON-like layout (text allowed inside values):
{
  "title": "Value Plan",
  "recommended_allocation": {"equity_percent": <int>, "debt_percent": <int>, "cash_percent": <int>},
  "top_picks": [
    {"name": "...", "type": "Value MF | Large-cap Stock | Dividend ETF | Debt Instrument", "rationale": "..."},
    {"name": "..."},
    {"name": "..."}
  ],
  "actionable_next_steps": ["..."],
  "risk_notes": "...",
  "explain_short": "1-2 sentence justification"
}

Use Indian market context and keep allocations summing to ~100%.
End with TERMINATE.
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=termination_for("value_agent"),
)


# Investment Advisor Agent

In [47]:
investment_advisor_agent = ConversableAgent(
    name="investment_advisor_agent",
    system_message="""
You are the Investment Advisor Agent. Compile the final personalized report using the full chat history.

Report structure:
# Personalized Financial Investment Report
## Executive Summary
## Portfolio Analysis Summary
## Investment Strategy
## Detailed Investment Recommendations
## Action Plan (Immediate 0-30d, Short-term 30-90d, Long-term 90-365d)
## Risk Assessment & Mitigation
## Monitoring & Review Schedule
## Conclusion

Be concise but comprehensive. Use Indian financial context. End with TERMINATE.
""",
    description="Synthesizes prior agents into a final report.",
    llm_config=llm_config,
    human_input_mode="NEVER",
    is_termination_msg=termination_for("investment_advisor_agent"),
)

In [48]:
def extract_recommendation(message: str) -> Optional[str]:
    """Extract Growth/Value/Hybrid from an agent message."""
    if not message:
        return None

    pattern = r"RECOMMENDATION:\s*(Growth|Value|Hybrid)"
    match = re.search(pattern, message, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()

    # Try JSON parsing as fallback
    try:
        data = json.loads(message)
        rec = data.get("recommendation")
        if isinstance(rec, str) and rec.lower() in {"growth", "value", "hybrid"}:
            return rec.capitalize()
    except Exception:
        pass

    return None

In [49]:
# Track hybrid flow so we can route Growth -> Value -> Advisor
flow_state: Dict[str, Any] = {
    "hybrid_mode": False,
    "growth_done": False,
    "value_done": False,
}



In [50]:
def stateflow_router(last_speaker: Any, groupchat: GroupChat) -> Optional[ConversableAgent]:
    """
    Minimal StateFlow-like routing:
    1) Start -> portfolio_analysis_agent
    2) After portfolio_analysis_agent -> growth_agent or value_agent (Hybrid triggers both)
    3) After growth/value -> investment_advisor_agent (Hybrid: Growth -> Value -> Advisor)
    4) After investment_advisor_agent -> terminate
    """
    messages = groupchat.messages
    if not messages:
        # Reset flow state at the start of a conversation
        flow_state["hybrid_mode"] = False
        flow_state["growth_done"] = False
        flow_state["value_done"] = False
        return portfolio_analysis_agent

    last_msg = messages[-1]
    sender = (last_msg.get("name") if isinstance(last_msg, dict) else "") or ""
    content = (last_msg.get("content") if isinstance(last_msg, dict) else "") or ""
    sender_lower = sender.lower()

    if "portfolio_analysis_agent" in sender_lower or last_speaker == portfolio_analysis_agent:
        rec = extract_recommendation(content)
        if rec and rec.lower() == "hybrid":
            flow_state["hybrid_mode"] = True
            flow_state["growth_done"] = False
            flow_state["value_done"] = False
            return growth_agent
        flow_state["hybrid_mode"] = False
        if rec == "Value":
            return value_agent
        # Default to growth when Hybrid or missing
        return growth_agent

    if "growth_agent" in sender_lower or last_speaker == growth_agent:
        if flow_state.get("hybrid_mode"):
            flow_state["growth_done"] = True
            # Route to Value next if not already done
            if not flow_state.get("value_done"):
                return value_agent
        return investment_advisor_agent

    if "value_agent" in sender_lower or last_speaker == value_agent:
        if flow_state.get("hybrid_mode"):
            flow_state["value_done"] = True
        return investment_advisor_agent

    if "investment_advisor_agent" in sender_lower or last_speaker == investment_advisor_agent:
        return None

    return portfolio_analysis_agent

In [51]:
# --- Group chat wiring ---
group_chat = GroupChat(
    agents=[portfolio_analysis_agent, growth_agent, value_agent, investment_advisor_agent],
    messages=[],
    max_round=15,
    speaker_selection_method=stateflow_router,
    allow_repeat_speaker=False,
)

group_chat_manager = GroupChatManager(groupchat=group_chat, llm_config=llm_config)



In [52]:
def run_portfolio_workflow(
    salary: str,
    portfolio_summary: str,
    investment_horizon_years: Optional[int] = None,
    risk_tolerance: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Execute the portfolio manager workflow and return the agent outputs.
    """
    user_message = (
        "I want help managing my investments.\n"
        f"Salary: {salary}\n"
        f"Portfolio: {portfolio_summary}\n"
        f"Investment horizon (years): {investment_horizon_years}\n"
        f"Risk tolerance: {risk_tolerance}\n"
        "Please analyze and recommend whether I should pursue Growth or Value investing."
    )

    user_proxy.initiate_chat(
        group_chat_manager,
        message=user_message,
        max_turns=5,
        clear_history=True,
        summary_method="last_msg",
    )

    outputs: Dict[str, Any] = {
        "portfolio_analysis": "",
        "recommendation": None,
        "selected_strategy": None,  # Growth | Value | Hybrid
        "investment_recommendations": "",
        "final_report": "",
        "messages": list(group_chat.messages),
    }

    rec = None
    hybrid_recos = []

    for msg in group_chat.messages:
        if not isinstance(msg, dict):
            continue
        sender = msg.get("name", "")
        content = msg.get("content", "")

        if "portfolio_analysis_agent" in sender:
            outputs["portfolio_analysis"] = content
            rec = extract_recommendation(content)
            outputs["recommendation"] = rec
        elif "growth_agent" in sender:
            if outputs["selected_strategy"] not in {"Value", "Hybrid"}:
                outputs["selected_strategy"] = "Growth"
            if rec and rec.lower() == "hybrid":
                hybrid_recos.append({"agent": "growth", "content": content})
            else:
                outputs["investment_recommendations"] = content
        elif "value_agent" in sender:
            if outputs["selected_strategy"] not in {"Growth", "Hybrid"}:
                outputs["selected_strategy"] = "Value"
            if rec and rec.lower() == "hybrid":
                outputs["selected_strategy"] = "Hybrid"
                hybrid_recos.append({"agent": "value", "content": content})
            else:
                outputs["investment_recommendations"] = content
        elif "investment_advisor_agent" in sender:
            outputs["final_report"] = content

    if rec and rec.lower() == "hybrid":
        outputs["selected_strategy"] = "Hybrid"
        # Concatenate both recommendations for convenience
        if hybrid_recos:
            combined = []
            for item in hybrid_recos:
                combined.append(f"[{item['agent']}] {item['content']}")
            outputs["investment_recommendations"] = "\n\n".join(combined)

    return outputs


In [53]:
demo_result = run_portfolio_workflow(
    salary="₹2.15 lakhs/month",
    portfolio_summary="SIP ₹30k/mo (2y), FD ₹3L, Cash ₹1.5L, EMI ₹52k/mo, EPF ongoing",
    investment_horizon_years=7,
    risk_tolerance="moderate",
)

print("\n" + "=" * 80)
print("FINAL PERSONALIZED REPORT")
print("=" * 80)
print(demo_result.get("final_report") or "Report not generated.")

user_proxy (to chat_manager):

I want help managing my investments.
Salary: ₹2.15 lakhs/month
Portfolio: SIP ₹30k/mo (2y), FD ₹3L, Cash ₹1.5L, EMI ₹52k/mo, EPF ongoing
Investment horizon (years): 7
Risk tolerance: moderate
Please analyze and recommend whether I should pursue Growth or Value investing.

--------------------------------------------------------------------------------

Next speaker: portfolio_analysis_agent

portfolio_analysis_agent (to chat_manager):

**Salary Analysis:**  
Your monthly salary of ₹2.15 lakhs gives you a solid income base. After accounting for your monthly EMI of ₹52k, your disposable income appears to be around ₹1.63 lakhs. This can be effectively utilized for investments or saving towards financial goals.

**Portfolio Strength Breakdown:**  
- **SIP:** ₹30k/month (Total: ₹7.2 lakhs over 2 years)
- **FD:** ₹3 lakhs (Safe but lower returns)
- **Cash:** ₹1.5 lakhs (Liquidity available for emergencies or opportunities)
- **EMI:** ₹52k/month (Annual burden o

In [55]:
from IPython.display import display, Markdown
agent_name_to_find = "investment_advisor_agent"

# Iterate through the dictionary to find the last conversation for the specified agent
for agent, interactions in group_chat_manager.chat_messages.items():
    # Filter interactions by agent name
    filtered_interactions = [i for i in interactions if i.get("name") == agent_name_to_find]
    if filtered_interactions:
        # Print the last conversation's content
        display(Markdown(filtered_interactions[-1]["content"]))
        break
else:
    print("Agent not found or no interactions available.")

# Personalized Financial Investment Report

## Executive Summary
This report outlines a tailored investment strategy to align with your financial goals, risk tolerance, and current portfolio structure. Your solid income base provides a comfortable platform for investment growth, ensuring long-term financial security while maintaining a moderate risk profile.

## Portfolio Analysis Summary
- **Salary:** ₹2.15 lakhs/month
- **Monthly Disposable Income:** ₹1.63 lakhs post-EMI
- **Current Investments:** 
  - SIP: ₹30k/month (Total of ₹7.2 lakhs over 2 years)
  - Fixed Deposit (FD): ₹3 lakhs
  - Cash Reserves: ₹1.5 lakhs
  - Ongoing EPF contributions
- **Risk Tolerance:** Moderate

## Investment Strategy
Based on your risk profile and financial circumstances, **Hybrid Investing** is recommended. This combines growth potential through equities with safer asset exposure, tailored to your moderate risk tolerance.

## Detailed Investment Recommendations

### Growth Plan (60% Equity Allocation)
- **Recommended Allocation:**
  - Equity: 60%
  - Debt: 30%
  - Cash: 10%
- **Top Picks:**
  1. **Nippon India Growth Fund** (Large-cap MF): Focuses on growth stocks with strong historical performance.
  2. **HDFC Mid-cap Opportunities Fund** (Mid-cap MF): Leverages high growth potential mid-cap companies.
  3. **ICICI Prudential Corporate Bond Fund** (Debt MF): Provides income and stability with corporate bond exposure.

### Value Plan (40% Equity Allocation)
- **Recommended Allocation:**
  - Equity: 40%
  - Debt: 40%
  - Cash: 20%
- **Top Picks:**
  1. **HDFC Equity Fund** (Value MF): Strong fundamentals focusing on large-cap stocks for steady growth.
  2. **SBI Magnum Fixed Deposit** (Debt Instrument): Better interest rates ensuring safety and regular returns.
  3. **Nippon India Nifty BeES** (Dividend ETF): Exposure to dividend yielding Nifty 50 companies for steady income.

## Action Plan
- **Immediate (0-30 days):**
  - Increase SIP investments to ₹40k/month (Growth Plan).
  - Review existing SIP to include mid-cap and large-cap mutual funds or value funds as appropriate.
  
- **Short-term (30-90 days):**
  - Adjust the cash component based on market conditions and upcoming financial needs.
  - Regularly monitor the performance of selected funds.

- **Long-term (90-365 days):**
  - Maintain discipline in reviewing the investment portfolio every quarter.
  - Explore additional avenues for investment as cash reserves grow from business or bonuses.

## Risk Assessment & Mitigation
Maintain a 10-20% liquidity (cash) component to capitalize on market corrections without compromising long-term growth. Regularly assess and balance between equity and debt investments to align with market trends and personal financial goals.

## Monitoring & Review Schedule
- Review your investment portfolio quarterly.
- Adjust allocations based on fresh market insights, financial goals, and risk tolerance adjustments.

## Conclusion
The proposed hybrid investment strategy leverages both growth and value aspects to maximize returns while maintaining a balanced approach to risk. This position will help you strategically build wealth over time and securely reach your financial ambitions.

TERMINATE.